# DeepScaler: train trên Google Colab / Kaggle

Notebook này **huấn luyện** DeepScaler (AdapGLA + học đồ thị thích nghi, Sec. IV-B/IV-C của báo) cho **Online Boutique**:
- Dữ liệu: 10 service × 5 đặc trưng `pod, cpu, res, req, mem` (Fig. 4 của báo).
- Mục tiêu: số pod ở bước kế tiếp (Sec. III: `y_t = x_{t+1}^{pod}`).

Notebook train nhiều seed, (tuỳ chọn) train thêm các model so sánh của Fig. 12 (DCRNN, T-GCN), in bảng tổng hợp và tải toàn bộ model về.

**Cách dùng**
1. *Runtime → Change runtime type → **T4 GPU*** (Kaggle: *Accelerator → GPU*, bật *Internet*).
2. Sửa ô **CẤU HÌNH** nếu cần, rồi *Runtime → **Run all***.
3. Ở bước 1 notebook sẽ hỏi upload: chọn **cùng lúc** file CSV (vd. `run2_hpa_5f.csv`) và file call graph `*_od.npy` (vd. `run2_hpa_od.npy`). Hai file này nằm ở `data/raw/` trên máy bạn.
4. Ở bước cuối, Chrome sẽ tải về `deepscaler_models.zip`. Giải nén zip vào thư mục repo. Thư mục `model_states/ds_s<seed>` dùng được ngay cho `deploy/04_run_deepscaler.sh` (`MODEL_DIR=model_states/ds_s2021`).

**Nguồn dữ liệu** (`DATA_SOURCE`):
- `"cluster"`: CSV thật thu bằng `pipeline/collect.py` trên Kubernetes + Istio (`deploy/RUN_ONLINE.md`).
- `"simulated"`: dữ liệu **mô phỏng** (`pipeline/simulate.py`), chỉ để thử pipeline, **không** dùng làm kết quả luận văn.

Mỗi lần train mất khoảng 20 s trên T4. Toàn bộ (3 seed DeepScaler + 9 model so sánh) mất khoảng 5–8 phút.

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL    = "https://github.com/maintn24/DeepScaler.git"
BRANCH      = "deepscaler-pipeline"
DATA_SOURCE = "cluster"        # "cluster" (dữ liệu thật) | "simulated" (chỉ để thử pipeline)
RAW_CSV     = None             # None = hỏi upload trên Colab; Kaggle: "/kaggle/input/<dataset>/run2_hpa_5f.csv"
OD_ADJ      = None             # None = lấy file *_od.npy upload cùng CSV; không có thì dùng call graph tĩnh của Boutique
DATA_NAME   = "c20"            # tên dataset -> data/<DATA_NAME>/
SPLIT       = "0.6,0.2,0.2"    # train/valid/test, chia theo thời gian
STEP_IN     = 20               # số bước lịch sử (20 x 30 s = 10 phút). Repo gốc dùng 80
FEATURES    = "pod,cpu,res,req,mem"   # đặc trưng đầu vào, 'pod' phải đứng đầu (Fig. 4 của báo)

# --- DeepScaler (train.py) ---
SEEDS       = [2021, 2022, 2023]
NUM_EPOCH   = 15               # số epoch tối đa mỗi vòng (repo gốc: main.py 1, predict_scale.py 5)
NUM_ITER    = 6                # số vòng luân phiên predictor <-> graph learner (Algorithm 2)
ADJ         = "od,cc"          # đồ thị khởi tạo: od = call graph, cc = tương quan request
EPSILON     = None             # ngưỡng làm thưa đồ thị (eq. 6). None = mặc định của code (0.5/N)
DELTA       = None             # biên độ chênh đồ thị trong loss (eq. 7). None = mặc định 0.05

# --- Model so sánh của Fig. 12 (train_baselines.py) ---
RUN_BASELINES = True
BASELINES     = ["DCRNN", "TGCN", "LSTM"]   # DCRNN [49], T-GCN [50]; LSTM không có trong báo

SIM_LOOPS   = 16               # chỉ dùng khi DATA_SOURCE = "simulated"

In [ ]:
import os, sys, subprocess, glob, json, re
IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB  = "google.colab" in sys.modules
WORK = "/kaggle/working" if IN_KAGGLE else "/content"
os.chdir(WORK)
if os.path.exists("DeepScaler"):
    subprocess.run(["git", "-C", "DeepScaler", "pull", "-q"], check=False)     # lấy code mới nhất
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, "DeepScaler"], check=True)
os.chdir(f"{WORK}/DeepScaler")
for f in ["train.py", "train_baselines.py", "pipeline/build_dataset.py"]:
    assert os.path.exists(f), f"Repo thiếu {f}: hãy push code mới nhất lên nhánh {BRANCH}"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| commit:",
      subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
if not torch.cuda.is_available():
    print("CẢNH BÁO: không có GPU -> vẫn chạy được trên CPU nhưng chậm hơn nhiều")


def run(cmd, log=None, tail=6):
    """Chạy lệnh (không qua shell -> không bị IPython/bash thay $ hay {}), lưu log, in vài dòng cuối."""
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    out = r.stdout + r.stderr
    if log:
        os.makedirs(os.path.dirname(log), exist_ok=True)
        open(log, "w").write(out)
    lines = [l for l in out.splitlines() if l.strip() and "Warning" not in l and "warn(" not in l]
    print("\n".join(lines[-tail:]))
    if r.returncode != 0:
        raise RuntimeError(f"lệnh lỗi (mã {r.returncode}): {' '.join(map(str, cmd))}\n--- log ---\n{out[-3000:]}")
    return out

## 1. Dữ liệu thô

In [ ]:
if DATA_SOURCE == "simulated":
    RAW_CSV = "data/raw/simulated.csv"
    run([sys.executable, "-m", "pipeline.simulate", "--loops", SIM_LOOPS, "--interval", 30, "--out", RAW_CSV])
elif RAW_CSV is None and IN_COLAB:
    from google.colab import files
    print("Chọn CÙNG LÚC file CSV và file *_od.npy (giữ Ctrl để chọn nhiều file)")
    up = files.upload()
    csvs = [k for k in up if k.endswith(".csv")]
    npys = [k for k in up if k.endswith(".npy")]
    assert csvs, "chưa upload file .csv"
    RAW_CSV = csvs[0]
    if OD_ADJ is None and npys:
        OD_ADJ = npys[0]
assert RAW_CSV and os.path.exists(RAW_CSV), "Chỉ định RAW_CSV (Kaggle: /kaggle/input/<dataset>/<file>.csv)"
if OD_ADJ is None:
    print("Không có *_od.npy -> dùng call graph tĩnh của Online Boutique (pipeline/common.py)")

import pandas as pd, numpy as np
raw = pd.read_csv(RAW_CSV)
print(f"{RAW_CSV}: {raw.shape[0]} bước x {raw.shape[1]} cột | OD: {OD_ADJ}")
if "unix_time" in raw:
    print("thời gian:", pd.to_datetime(raw.unix_time.iloc[0], unit="s"), "->", pd.to_datetime(raw.unix_time.iloc[-1], unit="s"),
          f"(UTC, bước {np.median(np.diff(raw.unix_time)):.0f} s)")
raw.filter(regex="_pod$").describe().loc[["mean", "min", "max"]].round(2)

## 2. Tạo tập train / valid / test (chia theo thời gian theo `SPLIT`)

In [ ]:
DATA_DIR = f"data/{DATA_NAME}"
cmd = [sys.executable, "-m", "pipeline.build_dataset", "--raw", RAW_CSV, "--out_dir", DATA_DIR,
       "--step_in", STEP_IN, "--split", SPLIT, "--features", FEATURES]
if OD_ADJ:
    cmd += ["--od_adj", OD_ADJ]
run(cmd, tail=5)
meta = json.load(open(f"{DATA_DIR}/meta.json"))
print("features:", meta["features"], "| samples:", meta["samples"])

## 3. Chạy thử nhanh (1 epoch, 1 vòng) để chắc pipeline không lỗi

In [ ]:
run([sys.executable, "train.py", "--data_dir", DATA_DIR, "--out_dir", "model_states/smoke",
     "--num_epoch", 1, "--num_iter", 1], log="logs/smoke.log", tail=4)

## 4. Huấn luyện DeepScaler (mỗi seed một thư mục `model_states/ds_s<seed>`)

Seed được truyền qua một bản sao của `config/train_config.yaml`, file gốc không bị sửa.

In [ ]:
import yaml, time
base_cfg = yaml.safe_load(open("config/train_config.yaml"))
DS_DIRS = []
for seed in SEEDS:
    cfg_path = f"/tmp/train_config_s{seed}.yaml"
    yaml.safe_dump({**base_cfg, "seed": seed}, open(cfg_path, "w"))
    out_dir = f"model_states/ds_s{seed}"
    cmd = [sys.executable, "train.py", "--data_dir", DATA_DIR, "--out_dir", out_dir,
           "--train_config_path", cfg_path, "--num_epoch", NUM_EPOCH, "--num_iter", NUM_ITER, "--adj", ADJ]
    if EPSILON is not None: cmd += ["--epsilon", EPSILON]
    if DELTA is not None:   cmd += ["--delta", DELTA]
    t0 = time.time(); print(f"=== DeepScaler seed {seed} ===")
    run(cmd, log=f"logs/ds_s{seed}.log", tail=4)
    print(f"({time.time() - t0:.0f} s)\n")
    DS_DIRS.append(out_dir)

## 5. (Tuỳ chọn) Model so sánh của Fig. 12

Mỗi model chạy trên cùng dữ liệu, cùng cách chia tập, cùng chuẩn hoá và metric với DeepScaler. Đồ thị cố định là call graph (Sec. V-C: *"only use predefined affinity matrices"*).

In [ ]:
BL_DIRS = []
if RUN_BASELINES:
    for m in BASELINES:
        for seed in SEEDS:
            out_dir = f"model_states/bl_{m}_s{seed}"
            t0 = time.time(); print(f"=== {m} seed {seed} ===")
            run([sys.executable, "train_baselines.py", "--data_dir", DATA_DIR, "--model", m,
                 "--seed", seed, "--out_dir", out_dir], log=f"logs/bl_{m}_s{seed}.log", tail=1)
            print(f"({time.time() - t0:.0f} s)")
            BL_DIRS.append(out_dir)

## 6. Kết quả trên tập test (trung bình ± độ lệch chuẩn qua các seed)

In [ ]:
def load(d): return json.load(open(f"{d}/metrics.json"))
COLS = ["mae", "rmse", "mape", "pods_exact_match", "under_provision_rate", "over_provision_rate"]
groups = {"DeepScaler (AdapGLA)": [load(d)["results"]["test"]["DeepScaler(AdapGLA)"] for d in DS_DIRS]}
for d in BL_DIRS:
    m = load(d); groups.setdefault(m["model"], []).append(m["results"]["test"][m["model"]])
groups["Persistence (giữ số pod bước trước)"] = [load(DS_DIRS[0])["results"]["test"]["Persistence(last pods)"]]
rows = {}
for name, rs in groups.items():
    rows[name] = {c: f"{np.mean([r[c] for r in rs]):.3f} ± {np.std([r[c] for r in rs]):.3f}" for c in COLS}
    rows[name]["n"] = len(rs)
table = pd.DataFrame(rows).T[["n"] + COLS]
table.to_csv("logs/summary_test.csv")
G = [load(d)["graph"] for d in DS_DIRS]
print("Đồ thị DeepScaler học được: số cạnh", [g["learned_edges(off-diag)"] for g in G],
      "| similarity với call graph (Fig. 11, báo: 0.89) OD %.2f, CC %.2f" % (
          np.mean([g.get("paper_similarity_od", np.nan) for g in G]), np.mean([g.get("paper_similarity_cc", np.nan) for g in G])))
table

In [ ]:
import matplotlib.pyplot as plt
P = np.load(f"{DS_DIRS[0]}/predictions.npz")
services = meta["services"]
show = ["frontend", "recommendationservice", "currencyservice", "checkoutservice"]
fig, axes = plt.subplots(len(show), 1, figsize=(11, 2.4 * len(show)), sharex=True)
for ax, s in zip(axes, show):
    j = services.index(s)
    ax.step(range(len(P["test_true"])), P["test_true"][:, 0, j], where="post", color="#2a78d6", lw=2, label="Pod thực tế")
    ax.plot(P["test"][:, 0, j], color="#eb6834", lw=1.5, label=f"DeepScaler ({DS_DIRS[0]})")
    ax.set_title(s, loc="left", fontsize=10); ax.grid(alpha=.25); ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(loc="upper right", frameon=False, ncol=2)
axes[-1].set_xlabel("Bước thời gian (tập test, 30 s/bước)")
plt.tight_layout(); plt.show()

In [ ]:
adj = np.load(f"{DS_DIRS[0]}/AdapGLA/best_adj_mx.npy"); od = np.load(f"{DATA_DIR}/adj_od.npy")
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
short = [s.replace("service", "") for s in services]
for a, M, t in [(ax[0], od + od.T, "Call graph (Istio)"), (ax[1], adj, f"Đồ thị học được ({DS_DIRS[0]})")]:
    im = a.imshow(M, cmap="Blues"); a.set_title(t, fontsize=10)
    a.set_xticks(range(len(services))); a.set_xticklabels(short, rotation=60, fontsize=8)
    a.set_yticks(range(len(services))); a.set_yticklabels(short, fontsize=8)
    fig.colorbar(im, ax=a, fraction=.046)
plt.tight_layout(); plt.show()

## 7. Tải toàn bộ model về

Zip gồm:
- `model_states/ds_s*`: model DeepScaler.
- `model_states/bl_*`: model so sánh.
- `data/<DATA_NAME>/meta.json` và hai file đồ thị.
- `logs/`: log train và bảng tổng hợp.

Giải nén vào thư mục repo trên máy bạn: `cd ~/Projects/DeepScaler && unzip -o ~/Downloads/deepscaler_models.zip`.

In [ ]:
import zipfile
zip_path = f"{WORK}/deepscaler_models.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for d in DS_DIRS + BL_DIRS:
        for f in glob.glob(f"{d}/**/*", recursive=True):
            if os.path.isfile(f): z.write(f)
    for f in ["meta.json", "adj_od.npy", "adj_cc.npy"]:
        z.write(f"{DATA_DIR}/{f}")
    for f in glob.glob("logs/*"):
        z.write(f)
print(zip_path, f"{os.path.getsize(zip_path) / 1e6:.1f} MB")
if IN_COLAB:
    from google.colab import files; files.download(zip_path)
# Kaggle: file nằm trong /kaggle/working -> tab Output